In [ ]:
%pip install -q keras-hub tensorflow

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import tensorflow as tf
import keras
****************
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
model = ****************.from_preset("deeplab_v3_plus_resnet50_pascalvoc")
converter = keras_hub.layers.DeepLabV3ImageConverter(image_size=(512, 512), interpolation="bilinear")
preprocessor = keras_hub.models.DeepLabV3ImageSegmenterPreprocessor(converter)

In [ ]:
url = "https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png"

path = keras.utils.get_file(origin=url)

image = np.array(keras.utils.load_img(path))

plt.figure(figsize=(8, 5))
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
****************
predictions = model.predict(tf.expand_dims(input_image, axis=0))
mask = np.argmax(predictions[0], axis=-1)

print("Image shape:", input_image.shape)
print("Mask shape:", mask.shape)
print("Predicted class IDs:", np.unique(mask))

In [ ]:
%pip install -q tensorflow-datasets

In [ ]:
import tensorflow_datasets as tfds

info = tfds.builder("voc/2012").info
****************

print("Predicted classes:", [voc_classes[i] for i in np.unique(mask)])

In [ ]:
original = np.clip(np.array(input_image) / 255.0, 0, 1)
colors = plt.get_cmap("tab20", 21)
colored_mask = colors(mask)[..., :3]
overlay = original.copy()
****************
overlay[foreground] = 0.55 * original[foreground] + 0.45 * colored_mask[foreground]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(original)
axes[0].set_title("Original Image")
axes[1].imshow(mask, cmap="tab20", vmin=0, vmax=20)
axes[1].set_title("Segmentation Mask")
axes[2].imshow(overlay)
axes[2].set_title("Segmentation Overlay")

for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
%pip install -q transformers torch torchvision pillow matplotlib requests

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import requests
from io import BytesIO
from PIL import Image
****************

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/segmentation_input.jpg"
image = Image.open(BytesIO(requests.get(url).content)).convert("RGB")

plt.figure(figsize=(9, 5))
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
****************

processor = AutoImageProcessor.from_pretrained(checkpoint)
model = Mask2FormerForUniversalSegmentation.from_pretrained(checkpoint).eval()

inputs = processor(images=image, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

result = processor.post_process_panoptic_segmentation(outputs, target_sizes=[(image.height, image.width)])[0]

print("Detected segments:", len(result["segments_info"]))

In [ ]:
import requests

url = "https://raw.githubusercontent.com/cocodataset/panopticapi/master/panoptic_coco_categories.json"
categories = requests.get(url).json()

****************

print(thing_names)

In [ ]:
panoptic = result["segmentation"].numpy()
semantic = np.zeros_like(panoptic)
instance = np.zeros_like(panoptic)

count = 0

for segment in result["segments_info"]:
    region = panoptic == segment["id"]
    label = segment["label_id"]
    name = model.config.id2label[label].lower()

    ****************

    if name in thing_names:
        count += 1
        ****************

print("Semantic classes:", np.unique(semantic))
print("Object instances:", count)
print("Panoptic segments:", len(result["segments_info"]))


In [ ]:
def colorize(mask):
    rgb = np.zeros((*mask.shape, 3))
    colors = plt.get_cmap("tab20")
    for i, value in enumerate(np.unique(mask)):
        if value != 0: rgb[mask == value] = colors(i % 20)[:3]
    return rgb

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].imshow(image)
axes[0, 0].set_title("Original Image")

axes[0, 1].imshow(colorize(semantic))
axes[0, 1].set_title("Semantic Segmentation")

axes[1, 0].imshow(colorize(instance))
axes[1, 0].set_title("Instance Segmentation")

axes[1, 1].imshow(colorize(panoptic))
axes[1, 1].set_title("Panoptic Segmentation")

for ax in axes.flat: ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
print("Detected Segments:\n")

for segment in result["segments_info"]:
    label = model.config.id2label[segment["label_id"]]
    print(f"Segment {segment['id']}: {label}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.imshow(colorize(panoptic))

for segment in result["segments_info"]:
    region = panoptic == segment["id"]
    y, x = np.where(region)
    if len(x) == 0: continue
    name = model.config.id2label[segment["label_id"]]
    plt.text(np.median(x), np.median(y), name, color="black", fontsize=9, ha="center", bbox=dict(facecolor="white", alpha=0.8))

plt.axis("off")
plt.title("Panoptic Segmentation with Class Labels")
plt.show()

In [ ]:
%pip install -q tensorflow tf-keras huggingface_hub matplotlib pillow

In [ ]:
import tensorflow as tf
import tf_keras
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

url = "https://raw.githubusercontent.com/nikhilroxtomar/Multiclass-Segmentation-in-Unet/master/model.h5"
path = tf.keras.utils.get_file("unet_pets.h5", origin=url)

model = tf_keras.models.load_model(path, compile=False)

print("Input:", model.input_shape)
print("Output:", model.output_shape)

In [ ]:
url = "https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg"
path = tf.keras.utils.get_file("labrador.jpg", origin=url)

image = np.array(Image.open(path).convert("RGB").resize((256, 256)))
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
input_image = image[..., ::-1].astype("float32") / 255.0
prediction = model.predict(input_image[None, ...])
mask = np.argmax(prediction[0], axis=-1)

print("Prediction shape:", prediction.shape)
print("Mask shape:", mask.shape)
print("Predicted classes:", np.unique(mask))

In [ ]:
pet_mask = mask == 0
overlay = image.copy()
overlay[pet_mask] = 0.5 * image[pet_mask] + 0.5 * np.array([0, 255, 0])

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(image)
axes[0].set_title("Original Image")
axes[1].imshow(pet_mask, cmap="gray")
axes[1].set_title("Predicted Dog Mask")
axes[2].imshow(overlay.astype("uint8"))
axes[2].set_title("Segmentation Overlay")

for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(mask, cmap="viridis", vmin=0, vmax=2)
plt.colorbar(ticks=[0, 1, 2], label="Predicted Class ID")
plt.title("U-Net Pixel Classification")
plt.axis("off")
plt.show()

In [ ]:
%pip install -q importlib_resources tensorflow-datasets

import tensorflow_datasets as tfds
import tensorflow as tf
import numpy as np

ds = tfds.load("oxford_iiit_pet", split="test", shuffle_files=False)
sample = next(iter(ds))

image = tf.image.resize(sample["image"], (256, 256)).numpy().astype("uint8")
true_mask = tf.image.resize(sample["segmentation_mask"], (256, 256), method="nearest").numpy()[:, :, 0] - 1

input_image = image[..., ::-1].astype("float32") / 255.0
prediction = model.predict(input_image[None, ...])
mask = np.argmax(prediction[0], axis=-1)

In [ ]:
import tensorflow as tf
import numpy as np

accuracy = tf.keras.metrics.Accuracy()
accuracy.update_state(true_mask.flatten(), mask.flatten())

miou = tf.keras.metrics.MeanIoU(num_classes=3)
****************

iou = tf.keras.metrics.IoU(num_classes=3, target_class_ids=[0])
iou.update_state(true_mask.flatten(), mask.flatten())

print(f"Pixel Accuracy: {accuracy.result().numpy():.3f}")
print(f"Pet IoU:        {iou.result().numpy():.3f}")
print(f"Mean IoU:       {miou.result().numpy():.3f}")

pet_iou = iou.result().numpy()
pet_dice = 2 * pet_iou / (1 + pet_iou)

print(f"Pet Dice: {pet_dice:.3f}")